# LoRA in Practice: From Concept to RK3588 Deployment

This notebook explains how LoRA is actually used in production systems.

Focus:
- What LoRA changes
- What LoRA does NOT change
- Real training workflow
- Merging adapters
- Converting to RKLLM
- Deployment on RK3588


## 1. Why LoRA Exists

Full fine-tuning modifies billions of parameters.

LoRA freezes the base model and trains only small low-rank matrices.

Result:

- Less VRAM
- Faster training
- Smaller artifacts
- Multiple personalities/domain adapters on same base model


## 2. Conceptual Diagram  

![lora](res/LoRA.png)

```plantuml
@startuml
rectangle "Base Model" as BASE
rectangle "LoRA Adapter" as LORA
rectangle "Inference" as INF

BASE --> INF
LORA --> INF
@enduml
```

The base model remains frozen.
The adapter injects behavior changes.

## 3. Good LoRA Use Cases

### Persona

- Teacher
- Poet
- Customer support

### Style

- Formal
- Scientific
- Storytelling

### Domain specialization

- Linux admin
- Embedded systems
- Medical coding

### Structured output

- JSON
- SQL
- YAML


## 4. Bad LoRA Use Cases

Do NOT use LoRA for rapidly changing facts.

Bad:

- Today's stock prices
- Latest news
- Company documentation updated weekly

Use RAG instead.

## 5. Training Dataset Example

Teaching a model to answer as a poet.

```json
[
  {
    "instruction": "Describe rain",
    "response": "Rain writes silver poems across the earth."
  },
  {
    "instruction": "Describe morning",
    "response": "Morning unfolds like a page of light."
  }
]
```

## 6. PEFT LoRA Example

```python
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-0.5B'
)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj','v_proj'],
    lora_dropout=0.05
)

model = get_peft_model(model, config)
```

Only the adapter is trainable.

## 7. What Gets Saved

Base model:

```text
Qwen2.5-0.5B
```

Adapter:

```text
adapter_model.safetensors
adapter_config.json
```

Adapters are often only a few MB.

## 8. Inference With Adapter

```python
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    'my_lora'
)
```

Runtime combines:

Base Model + Adapter

## 9. Merge Adapter

Before RKLLM conversion:

```python
merged_model = model.merge_and_unload()
merged_model.save_pretrained('merged')
```

After merge, adapter weights become part of the model.

## 10. RK3588 Deployment Pipeline  

![lora-hg](res/LoRA-HG.png)

```plantuml
@startuml

rectangle GPU
rectangle LoRA
rectangle Merge
rectangle HuggingFace
rectangle RKLLMToolkit
rectangle RK3588

GPU --> LoRA
LoRA --> Merge
Merge --> HuggingFace
HuggingFace --> RKLLMToolkit
RKLLMToolkit --> RK3588

@enduml
```

Typical workflow:

1. Train on GPU.
2. Save adapter.
3. Merge adapter.
4. Convert to RKLLM.
5. Deploy to RK3588.

## 11. Multiple Adapters

One base model can support:

- poet-lora
- coding-lora
- linux-lora
- teacher-lora

This is much smaller than storing multiple full models.

## 12. LoRA + RAG

Production architecture:  

![RAG-LoRA](res/RAG-LoRA.png)

```plantuml
@startuml

actor User
rectangle RAG
rectangle LoRA
rectangle Model

User --> RAG
RAG --> Model
LoRA --> Model
Model --> User

@enduml
```

RAG = knowledge.

LoRA = behavior.

Model = reasoning.

## Key Takeaway

Use LoRA when you want to change HOW the model responds.

Use RAG when you want to change WHAT the model knows.

Most production systems use both.